# Day 3: Batch Ingestion (X12 EDI Claims & Debezium CDC)

This notebook simulates parsing synthetic raw EDI lines and Debezium JSON records into structured Bronze output.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit, expr, explode, split, from_json, col, when
from pyspark.sql.types import StructType, StructField, StringType, LongType
import os

warehouse_path = os.path.join(os.getcwd(), "spark-warehouse")

spark = SparkSession.builder \
    .appName("Day03_Batch_CDC") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", warehouse_path) \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark session created.")

## 1. X12 EDI Batch Processing Simulator

In [ ]:
x12_data = ["ISA*00*          *00*          *ZZ*1234567890     *ZZ*9876543210     *201201*1230*U*00501*000000001*0*T*:~GS*HC*1234567890*9876543210*20201201*1230*1*X*005010X222A1~"]
raw_df = spark.createDataFrame([(x,) for x in x12_data], ["value"])

segments_df = raw_df.select(explode(split(col("value"), "~")).alias("segment")) \
    .filter(col("segment") != "")

x12_processed_df = segments_df.selectExpr("segment as raw_payload_json") \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_system_id", lit("x12_batch_simulator")) \
    .withColumn("payload_id", expr("uuid()"))

x12_processed_df.write \
    .format("iceberg") \
    .mode("append") \
    .saveAsTable("local.ehdip_bronze.raw_payloads")

print("X12 records ingested:")
spark.sql("SELECT * FROM local.ehdip_bronze.raw_payloads WHERE source_system_id = 'x12_batch_simulator'").show(truncate=False)

## 2. Debezium CDC Processing Simulator

In [ ]:
cdc_json_data = [
    '{"payload": {"before": null, "after": "{\\"id\\": 1, \\"status\\": \\"created\\"}", "op": "c", "ts_ms": 1670000000000}}',
    '{"payload": {"before": "{\\"id\\": 1, \\"status\\": \\"created\\"}", "after": "{\\"id\\": 1, \\"status\\": \\"updated\\"}", "op": "u", "ts_ms": 1670000100000}}',
    '{"payload": {"before": "{\\"id\\": 2, \\"status\\": \\"active\\"}", "after": null, "op": "d", "ts_ms": 1670000200000}}'
]
cdc_df = spark.createDataFrame([(x,) for x in cdc_json_data], ["value"])

debezium_schema = StructType([
    StructField("payload", StructType([
        StructField("before", StringType()),
        StructField("after", StringType()),
        StructField("op", StringType()),
        StructField("ts_ms", LongType())
    ]))
])

parsed_df = cdc_df.select(from_json("value", debezium_schema).alias("data")).select("data.payload.*")

cdc_processed_df = parsed_df.withColumn("raw_payload_json", 
                                    when(col("op") == "d", col("before")).otherwise(col("after"))) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_system_id", lit("debezium_cdc_simulator")) \
    .withColumn("payload_id", expr("uuid()")) \
    .select("ingestion_timestamp", "source_system_id", "payload_id", "raw_payload_json")

cdc_processed_df.write \
    .format("iceberg") \
    .mode("append") \
    .saveAsTable("local.ehdip_bronze.raw_payloads")

print("CDC records ingested:")
spark.sql("SELECT * FROM local.ehdip_bronze.raw_payloads WHERE source_system_id = 'debezium_cdc_simulator'").show(truncate=False)